In [ ]:
!pip install -q transformers torch datasets scikit-learn pandas numpy matplotlib seaborn tqdm gradio accelerate


In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import BertForSequenceClassification, BertTokenizerFast, get_linear_schedule_with_warmup

warnings.filterwarnings("ignore")

SARCASM_DATA_PATH = "train-balanced-sarcasm.csv"
SENTIMENT_DATA_PATH = "Reddit_Data.csv"
SARCASM_CHECKPOINT_DIR = "./checkpoints/sarcasm"
SENTIMENT_CHECKPOINT_DIR = "./checkpoints/sentiment"
METRICS_PATH = "./checkpoints/metrics.json"
PERFORMANCE_DASHBOARD_PATH = "performance_summary.png"

MODEL_NAME = "bert-base-uncased"
MAX_TOKEN_LENGTH = 128
BATCH_SIZE = 16
EPOCH_COUNT = 3
LEARNING_RATE = 2e-5
RANDOM_SEED = 42
SARCASM_SAMPLE_SIZE = 20_000

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

for checkpoint_dir in (SARCASM_CHECKPOINT_DIR, SENTIMENT_CHECKPOINT_DIR):
    Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)

print(f"Device  : {DEVICE}")
print(f"PyTorch : {torch.__version__}")


In [ ]:
class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=MAX_TOKEN_LENGTH):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        item = {name: values[index] for name, values in self.encodings.items()}
        item["labels"] = self.labels[index]
        return item


def move_batch_to_device(batch):
    return {name: value.to(DEVICE) for name, value in batch.items()}


def train_one_epoch(model, data_loader, optimizer, scheduler):
    model.train()
    running_loss = 0.0

    for batch in tqdm(data_loader, desc="train", ncols=80, leave=False):
        batch = move_batch_to_device(batch)
        optimizer.zero_grad()

        output = model(**batch)
        output.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()
        running_loss += output.loss.item()

    return running_loss / len(data_loader)


@torch.no_grad()
def evaluate_model(model, data_loader):
    model.eval()
    predictions = []
    true_labels = []

    for batch in tqdm(data_loader, desc="eval", ncols=80, leave=False):
        labels = batch["labels"].tolist()
        batch = move_batch_to_device(batch)
        logits = model(**{key: value for key, value in batch.items() if key != "labels"}).logits

        predictions.extend(logits.argmax(dim=-1).cpu().tolist())
        true_labels.extend(labels)

    return (
        accuracy_score(true_labels, predictions),
        f1_score(true_labels, predictions, average="weighted"),
        true_labels,
        predictions,
    )


def fine_tune_classifier(model, train_loader, validation_loader, checkpoint_dir, epochs=EPOCH_COUNT):
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    training_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * training_steps),
        num_training_steps=training_steps,
    )

    best_f1_score = 0.0
    training_history = []

    for epoch in range(1, epochs + 1):
        print(f"\nEpoch {epoch}/{epochs}")
        training_loss = train_one_epoch(model, train_loader, optimizer, scheduler)
        validation_accuracy, validation_f1, _, _ = evaluate_model(model, validation_loader)

        print(
            f"loss={training_loss:.4f}  "
            f"val_acc={validation_accuracy:.4f}  "
            f"val_f1={validation_f1:.4f}"
        )
        training_history.append(
            {
                "epoch": epoch,
                "loss": training_loss,
                "val_acc": validation_accuracy,
                "val_f1": validation_f1,
            }
        )

        if validation_f1 > best_f1_score:
            best_f1_score = validation_f1
            model.save_pretrained(checkpoint_dir)
            print(f"Saved best checkpoint to {checkpoint_dir}")

    return training_history


def build_data_loader(texts, labels, tokenizer, shuffle=False):
    dataset = TextClassificationDataset(texts, labels, tokenizer)
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle)


print("Training helpers ready.")


## Model 1 - Sarcasm Detection

**Dataset:** `train-balanced-sarcasm.csv`
**Labels:** `0` Not Sarcastic, `1` Sarcastic


In [ ]:
print("[Sarcasm] Loading dataset")
sarcasm_df = pd.read_csv(SARCASM_DATA_PATH)[["label", "comment"]].dropna()
sarcasm_df["comment"] = sarcasm_df["comment"].astype(str).str.strip()
sarcasm_df = sarcasm_df[sarcasm_df["comment"].str.len() > 5].reset_index(drop=True)

print(f"Full dataset : {len(sarcasm_df):,} rows")
print(f"Label counts :\n{sarcasm_df['label'].value_counts().to_string()}")

if SARCASM_SAMPLE_SIZE:
    sample_size_per_class = SARCASM_SAMPLE_SIZE // 2
    sarcasm_df = pd.concat(
        [
            sarcasm_df[sarcasm_df["label"] == 0].sample(sample_size_per_class, random_state=RANDOM_SEED),
            sarcasm_df[sarcasm_df["label"] == 1].sample(sample_size_per_class, random_state=RANDOM_SEED),
        ]
    ).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    print(f"Using balanced sample of {len(sarcasm_df):,} rows")

sarcasm_texts = sarcasm_df["comment"].tolist()
sarcasm_labels = sarcasm_df["label"].tolist()

sarcasm_train_texts, sarcasm_holdout_texts, sarcasm_train_labels, sarcasm_holdout_labels = train_test_split(
    sarcasm_texts,
    sarcasm_labels,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=sarcasm_labels,
)
sarcasm_validation_texts, sarcasm_test_texts, sarcasm_validation_labels, sarcasm_test_labels = train_test_split(
    sarcasm_holdout_texts,
    sarcasm_holdout_labels,
    test_size=0.5,
    random_state=RANDOM_SEED,
    stratify=sarcasm_holdout_labels,
)

print(
    f"Split -> train={len(sarcasm_train_texts):,}  "
    f"val={len(sarcasm_validation_texts):,}  "
    f"test={len(sarcasm_test_texts):,}"
)


In [ ]:
print("[Sarcasm] Tokenizing data")
sarcasm_tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)

sarcasm_train_loader = build_data_loader(
    sarcasm_train_texts,
    sarcasm_train_labels,
    sarcasm_tokenizer,
    shuffle=True,
)
sarcasm_validation_loader = build_data_loader(
    sarcasm_validation_texts,
    sarcasm_validation_labels,
    sarcasm_tokenizer,
)
sarcasm_test_loader = build_data_loader(
    sarcasm_test_texts,
    sarcasm_test_labels,
    sarcasm_tokenizer,
)

print("Tokenization complete.")


In [ ]:
SARCASM_LABEL_NAMES = ["Not Sarcastic", "Sarcastic"]
SARCASM_ID_TO_LABEL = {0: "Not Sarcastic", 1: "Sarcastic"}
SARCASM_LABEL_TO_ID = {label: index for index, label in SARCASM_ID_TO_LABEL.items()}

if Path(SARCASM_CHECKPOINT_DIR, "config.json").exists():
    print(f"[Sarcasm] Loading checkpoint from {SARCASM_CHECKPOINT_DIR}")
    sarcasm_model = BertForSequenceClassification.from_pretrained(
        SARCASM_CHECKPOINT_DIR,
        num_labels=2,
    ).to(DEVICE)
    sarcasm_history = []
else:
    print("[Sarcasm] Starting fine-tuning")
    print("Tip: reduce SARCASM_SAMPLE_SIZE for a faster local run.")
    sarcasm_model = BertForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label=SARCASM_ID_TO_LABEL,
        label2id=SARCASM_LABEL_TO_ID,
    ).to(DEVICE)
    sarcasm_history = fine_tune_classifier(
        sarcasm_model,
        sarcasm_train_loader,
        sarcasm_validation_loader,
        SARCASM_CHECKPOINT_DIR,
    )

print("\n[Sarcasm] Test evaluation")
sarcasm_accuracy, sarcasm_f1, sarcasm_true_labels, sarcasm_predictions = evaluate_model(
    sarcasm_model,
    sarcasm_test_loader,
)
print(f"Accuracy : {sarcasm_accuracy:.4f}")
print(f"F1 (wtd) : {sarcasm_f1:.4f}\n")
print(classification_report(sarcasm_true_labels, sarcasm_predictions, target_names=SARCASM_LABEL_NAMES))


## Model 2 - Reddit Sentiment Analysis

**Dataset:** `Reddit_Data.csv`
**Labels:** `-1` Negative, `0` Neutral, `1` Positive


In [ ]:
print("[Sentiment] Loading dataset")
sentiment_df = pd.read_csv(SENTIMENT_DATA_PATH)[["clean_comment", "category"]].dropna()
sentiment_df["clean_comment"] = sentiment_df["clean_comment"].astype(str).str.strip()
sentiment_df = sentiment_df[sentiment_df["clean_comment"].str.len() > 5].reset_index(drop=True)

SENTIMENT_LABEL_NAMES = ["Negative", "Neutral", "Positive"]
SENTIMENT_REMAP = {-1: 0, 0: 1, 1: 2}

sentiment_df["label"] = sentiment_df["category"].map(SENTIMENT_REMAP)

print(f"Rows: {len(sentiment_df):,}")
print(
    "Classes:\n"
    + sentiment_df["label"]
    .value_counts()
    .rename({0: "Negative", 1: "Neutral", 2: "Positive"})
    .to_string()
)

sentiment_texts = sentiment_df["clean_comment"].tolist()
sentiment_labels = sentiment_df["label"].tolist()

sentiment_train_texts, sentiment_holdout_texts, sentiment_train_labels, sentiment_holdout_labels = train_test_split(
    sentiment_texts,
    sentiment_labels,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=sentiment_labels,
)
sentiment_validation_texts, sentiment_test_texts, sentiment_validation_labels, sentiment_test_labels = train_test_split(
    sentiment_holdout_texts,
    sentiment_holdout_labels,
    test_size=0.5,
    random_state=RANDOM_SEED,
    stratify=sentiment_holdout_labels,
)

print(
    f"Split -> train={len(sentiment_train_texts):,}  "
    f"val={len(sentiment_validation_texts):,}  "
    f"test={len(sentiment_test_texts):,}"
)


In [ ]:
print("[Sentiment] Tokenizing data")
sentiment_tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)

sentiment_train_loader = build_data_loader(
    sentiment_train_texts,
    sentiment_train_labels,
    sentiment_tokenizer,
    shuffle=True,
)
sentiment_validation_loader = build_data_loader(
    sentiment_validation_texts,
    sentiment_validation_labels,
    sentiment_tokenizer,
)
sentiment_test_loader = build_data_loader(
    sentiment_test_texts,
    sentiment_test_labels,
    sentiment_tokenizer,
)

print("Tokenization complete.")


In [ ]:
SENTIMENT_ID_TO_LABEL = {0: "Negative", 1: "Neutral", 2: "Positive"}
SENTIMENT_LABEL_TO_ID = {label: index for index, label in SENTIMENT_ID_TO_LABEL.items()}

if Path(SENTIMENT_CHECKPOINT_DIR, "config.json").exists():
    print(f"[Sentiment] Loading checkpoint from {SENTIMENT_CHECKPOINT_DIR}")
    sentiment_model = BertForSequenceClassification.from_pretrained(
        SENTIMENT_CHECKPOINT_DIR,
        num_labels=3,
    ).to(DEVICE)
    sentiment_history = []
else:
    print("[Sentiment] Starting fine-tuning")
    sentiment_model = BertForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=3,
        id2label=SENTIMENT_ID_TO_LABEL,
        label2id=SENTIMENT_LABEL_TO_ID,
    ).to(DEVICE)
    sentiment_history = fine_tune_classifier(
        sentiment_model,
        sentiment_train_loader,
        sentiment_validation_loader,
        SENTIMENT_CHECKPOINT_DIR,
    )

print("\n[Sentiment] Test evaluation")
sentiment_accuracy, sentiment_f1, sentiment_true_labels, sentiment_predictions = evaluate_model(
    sentiment_model,
    sentiment_test_loader,
)
print(f"Accuracy : {sentiment_accuracy:.4f}")
print(f"F1 (wtd) : {sentiment_f1:.4f}\n")
print(classification_report(sentiment_true_labels, sentiment_predictions, target_names=SENTIMENT_LABEL_NAMES))


## Performance Dashboard


In [ ]:
metrics = {
    "sarcasm": {
        "accuracy": round(sarcasm_accuracy, 4),
        "f1": round(sarcasm_f1, 4),
        "y_true": sarcasm_true_labels,
        "y_pred": sarcasm_predictions,
        "labels": SARCASM_LABEL_NAMES,
        "history": sarcasm_history,
    },
    "sentiment": {
        "accuracy": round(sentiment_accuracy, 4),
        "f1": round(sentiment_f1, 4),
        "y_true": sentiment_true_labels,
        "y_pred": sentiment_predictions,
        "labels": SENTIMENT_LABEL_NAMES,
        "history": sentiment_history,
    },
}

with open(METRICS_PATH, "w", encoding="utf-8") as metrics_file:
    json.dump(metrics, metrics_file, indent=2)

print(f"Metrics saved to {METRICS_PATH}")


In [ ]:
def plot_confusion_matrix(ax, true_labels, predictions, label_names, title):
    matrix = confusion_matrix(true_labels, predictions)
    matrix_percent = matrix.astype(float) / matrix.sum(axis=1, keepdims=True) * 100

    sns.heatmap(
        matrix_percent,
        annot=True,
        fmt=".1f",
        cmap="Blues",
        xticklabels=label_names,
        yticklabels=label_names,
        linewidths=0.5,
        ax=ax,
        cbar=False,
    )
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")


def plot_training_curve(ax, history, title):
    if not history:
        ax.text(
            0.5,
            0.5,
            "Loaded from\ncheckpoint\n(no history)",
            ha="center",
            va="center",
            transform=ax.transAxes,
            fontsize=11,
        )
    else:
        epochs = [item["epoch"] for item in history]
        ax.plot(epochs, [item["val_acc"] for item in history], "o-", label="Val Acc", color="#2196F3")
        ax.plot(epochs, [item["val_f1"] for item in history], "s--", label="Val F1", color="#4CAF50")
        ax.set_ylim(0, 1)
        ax.set_xlabel("Epoch")
        ax.legend()
        ax.grid(alpha=0.3)

    ax.set_title(title, fontweight="bold")


fig = plt.figure(figsize=(16, 11))
fig.suptitle("BERT Model Performance Summary", fontsize=16, fontweight="bold", y=1.01)
grid = gridspec.GridSpec(2, 3, figure=fig, hspace=0.5, wspace=0.4)

summary_ax = fig.add_subplot(grid[0, 0])
model_names = ["Sarcasm\nDetection", "Sentiment\nAnalysis"]
accuracy_scores = [sarcasm_accuracy, sentiment_accuracy]
f1_scores = [sarcasm_f1, sentiment_f1]
bar_positions = np.arange(len(model_names))

summary_ax.bar(bar_positions - 0.2, accuracy_scores, 0.35, label="Accuracy", color="#2196F3", alpha=0.85)
summary_ax.bar(bar_positions + 0.2, f1_scores, 0.35, label="F1 (wtd)", color="#4CAF50", alpha=0.85)
summary_ax.set_xticks(bar_positions)
summary_ax.set_xticklabels(model_names)
summary_ax.set_ylim(0, 1)
summary_ax.set_ylabel("Score")
summary_ax.set_title("Accuracy & F1 Comparison", fontweight="bold")
summary_ax.legend()
summary_ax.grid(axis="y", alpha=0.3)

for index, (accuracy, f1_score_value) in enumerate(zip(accuracy_scores, f1_scores)):
    summary_ax.text(index - 0.2, accuracy + 0.01, f"{accuracy:.3f}", ha="center", fontsize=9)
    summary_ax.text(index + 0.2, f1_score_value + 0.01, f"{f1_score_value:.3f}", ha="center", fontsize=9)

plot_training_curve(fig.add_subplot(grid[0, 1]), sarcasm_history, "Sarcasm - Training Curve")
plot_training_curve(fig.add_subplot(grid[0, 2]), sentiment_history, "Sentiment - Training Curve")
plot_confusion_matrix(
    fig.add_subplot(grid[1, :2]),
    sarcasm_true_labels,
    sarcasm_predictions,
    SARCASM_LABEL_NAMES,
    "Sarcasm - Confusion Matrix (%)",
)
plot_confusion_matrix(
    fig.add_subplot(grid[1, 2]),
    sentiment_true_labels,
    sentiment_predictions,
    SENTIMENT_LABEL_NAMES,
    "Sentiment - Confusion Matrix (%)",
)

plt.savefig(PERFORMANCE_DASHBOARD_PATH, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved dashboard to {PERFORMANCE_DASHBOARD_PATH}")


In [ ]:
import json
import sys
import tkinter as tk
from pathlib import Path
from tkinter import scrolledtext, ttk

import torch
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_CACHE = {}


def load_classifier(task_name):
    if task_name in MODEL_CACHE:
        return MODEL_CACHE[task_name]

    checkpoint_dir = SARCASM_CHECKPOINT_DIR if task_name == "sarcasm" else SENTIMENT_CHECKPOINT_DIR
    label_count = 2 if task_name == "sarcasm" else 3

    if not Path(checkpoint_dir, "config.json").exists():
        return None

    model = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir, num_labels=label_count)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    MODEL_CACHE[task_name] = (model.to(DEVICE).eval(), tokenizer)

    return MODEL_CACHE[task_name]


@torch.no_grad()
def predict_text_label(text, task_name, label_names):
    classifier = load_classifier(task_name)

    if classifier is None:
        required_cell = 6 if task_name == "sarcasm" else 9
        return None, f"Model not trained yet. Please run Cell {required_cell} first."

    cleaned_text = text.strip()
    if not cleaned_text:
        return None, "Please enter some text."

    model, tokenizer = classifier
    encoded_text = tokenizer(
        cleaned_text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_TOKEN_LENGTH,
    ).to(DEVICE)
    logits = model(**encoded_text).logits
    probabilities = F.softmax(logits, dim=-1).squeeze().cpu().tolist()

    if isinstance(probabilities, float):
        probabilities = [probabilities]

    predicted_index = int(logits.argmax(dim=-1).item())
    confidence_scores = {
        label_names[index]: round(probabilities[index] * 100, 1)
        for index in range(len(label_names))
    }

    return label_names[predicted_index], confidence_scores


class ConfidencePanel(tk.Frame):
    BAR_COLORS = {
        "Sarcastic": "#E53935",
        "Not Sarcastic": "#43A047",
        "Positive": "#1E88E5",
        "Neutral": "#FB8C00",
        "Negative": "#E53935",
    }

    def __init__(self, parent, label_names, **kwargs):
        super().__init__(parent, bg="#F5F5F5", **kwargs)
        self.rows = {}
        self.style = ttk.Style()

        for label_name in label_names:
            row = tk.Frame(self, bg="#F5F5F5")
            row.pack(fill="x", pady=3, padx=8)

            tk.Label(
                row,
                text=label_name,
                width=16,
                anchor="w",
                font=("Arial", 10),
                bg="#F5F5F5",
                fg="#333333",
            ).pack(side="left")

            progress_style = f"{label_name}.Horizontal.TProgressbar"
            progress_bar = ttk.Progressbar(
                row,
                length=260,
                maximum=100,
                mode="determinate",
                style=progress_style,
            )
            progress_bar.pack(side="left", padx=(4, 8))

            percentage_label = tk.Label(
                row,
                text="0.0%",
                width=6,
                anchor="e",
                font=("Arial", 10, "bold"),
                bg="#F5F5F5",
                fg="#555555",
            )
            percentage_label.pack(side="left")

            self.rows[label_name] = (progress_bar, percentage_label)
            self.style.configure(
                progress_style,
                troughcolor="#E0E0E0",
                background=self.BAR_COLORS.get(label_name, "#2196F3"),
                thickness=18,
            )

    def update_scores(self, scores):
        for label_name, (progress_bar, percentage_label) in self.rows.items():
            score = scores.get(label_name, 0.0)
            progress_bar["value"] = score
            percentage_label.config(text=f"{score:.1f}%")

    def reset(self):
        for progress_bar, percentage_label in self.rows.values():
            progress_bar["value"] = 0
            percentage_label.config(text="0.0%")


class ClassifierTab(tk.Frame):
    EXAMPLES = {
        "sarcasm": [
            "Oh great, another Monday. Just what I needed.",
            "I absolutely love sitting in traffic for two hours.",
            "The weather today is really nice and sunny.",
            "Yeah, because that plan worked SO well last time.",
        ],
        "sentiment": [
            "I really enjoyed reading about Buddhism. Very inspiring.",
            "The policy is neither good nor bad, just neutral.",
            "This is the worst experience I have ever had.",
            "Life is full of ups and downs, nothing special today.",
        ],
    }

    def __init__(self, parent, task_name, label_names, heading, action_text, **kwargs):
        super().__init__(parent, bg="#FAFAFA", **kwargs)
        self.task_name = task_name
        self.label_names = label_names

        tk.Label(
            self,
            text=heading,
            font=("Arial", 13, "bold"),
            bg="#FAFAFA",
            fg="#1F3864",
        ).pack(anchor="w", padx=16, pady=(14, 4))

        tk.Label(self, text="Input Text:", font=("Arial", 10), bg="#FAFAFA", fg="#444444").pack(anchor="w", padx=16)
        self.input_text = scrolledtext.ScrolledText(
            self,
            height=4,
            width=62,
            font=("Arial", 11),
            relief="solid",
            bd=1,
            wrap="word",
        )
        self.input_text.pack(padx=16, pady=(2, 8))

        button_row = tk.Frame(self, bg="#FAFAFA")
        button_row.pack(anchor="w", padx=16, pady=(0, 8))

        tk.Button(
            button_row,
            text=action_text,
            command=self.run_prediction,
            font=("Arial", 11, "bold"),
            bg="#2E75B6",
            fg="white",
            relief="flat",
            padx=18,
            pady=6,
            cursor="hand2",
            activebackground="#1F5496",
            activeforeground="white",
        ).pack(side="left")
        tk.Button(
            button_row,
            text="Clear",
            command=self.clear,
            font=("Arial", 11),
            bg="#E0E0E0",
            fg="#333333",
            relief="flat",
            padx=12,
            pady=6,
            cursor="hand2",
        ).pack(side="left", padx=(8, 0))

        self.result_text = tk.StringVar(value="-")
        tk.Label(
            self,
            textvariable=self.result_text,
            font=("Arial", 14, "bold"),
            bg="#FAFAFA",
            fg="#1F3864",
            anchor="w",
        ).pack(anchor="w", padx=16, pady=(0, 6))

        tk.Label(self, text="Confidence:", font=("Arial", 10, "bold"), bg="#FAFAFA", fg="#444444").pack(
            anchor="w",
            padx=16,
        )
        self.confidence_panel = ConfidencePanel(self, label_names)
        self.confidence_panel.pack(anchor="w", padx=8, pady=(2, 12))

        tk.Label(self, text="Quick examples:", font=("Arial", 10, "bold"), bg="#FAFAFA", fg="#444444").pack(
            anchor="w",
            padx=16,
        )
        example_row = tk.Frame(self, bg="#FAFAFA")
        example_row.pack(anchor="w", padx=16, pady=(2, 12))

        for example_text in self.EXAMPLES.get(task_name, []):
            tk.Button(
                example_row,
                text=example_text[:55] + ("..." if len(example_text) > 55 else ""),
                command=lambda selected_text=example_text: self.load_example(selected_text),
                font=("Arial", 9),
                bg="#E8F0FE",
                fg="#1A237E",
                relief="flat",
                padx=6,
                pady=3,
                cursor="hand2",
                wraplength=420,
                justify="left",
            ).pack(anchor="w", pady=2)

    def load_example(self, text):
        self.input_text.delete("1.0", "end")
        self.input_text.insert("1.0", text)
        self.run_prediction()

    def run_prediction(self):
        text = self.input_text.get("1.0", "end")
        predicted_label, result = predict_text_label(text, self.task_name, self.label_names)

        if predicted_label is None:
            self.result_text.set(f"Warning: {result}")
            self.confidence_panel.reset()
            return

        self.result_text.set(predicted_label)
        self.confidence_panel.update_scores(result)

    def clear(self):
        self.input_text.delete("1.0", "end")
        self.result_text.set("-")
        self.confidence_panel.reset()


class MetricsTab(tk.Frame):
    def __init__(self, parent, **kwargs):
        super().__init__(parent, bg="#FAFAFA", **kwargs)
        tk.Label(
            self,
            text="Model Performance Summary",
            font=("Arial", 13, "bold"),
            bg="#FAFAFA",
            fg="#1F3864",
        ).pack(anchor="w", padx=16, pady=(14, 8))

        self.text_area = scrolledtext.ScrolledText(
            self,
            height=18,
            width=62,
            font=("Courier", 10),
            state="disabled",
            relief="solid",
            bd=1,
            bg="#F0F4FF",
        )
        self.text_area.pack(padx=16, pady=(0, 10))

        tk.Button(
            self,
            text="Refresh Metrics",
            command=self.load_metrics,
            font=("Arial", 11, "bold"),
            bg="#2E75B6",
            fg="white",
            relief="flat",
            padx=14,
            pady=6,
            cursor="hand2",
            activebackground="#1F5496",
            activeforeground="white",
        ).pack(anchor="w", padx=16)

        self.load_metrics()

    def load_metrics(self):
        self.text_area.config(state="normal")
        self.text_area.delete("1.0", "end")

        if not Path(METRICS_PATH).exists():
            self.text_area.insert("end", "No metrics found. Train both models and run the metrics cell first.")
        else:
            with open(METRICS_PATH, encoding="utf-8") as metrics_file:
                metrics = json.load(metrics_file)

            sarcasm_metrics = metrics["sarcasm"]
            sentiment_metrics = metrics["sentiment"]
            lines = [
                "=" * 50,
                "  SARCASM DETECTION  (bert-base-uncased, binary)",
                "=" * 50,
                f"  Test Accuracy  :  {sarcasm_metrics['accuracy']:.4f}  ({sarcasm_metrics['accuracy'] * 100:.2f}%)",
                f"  Test F1 (wtd)  :  {sarcasm_metrics['f1']:.4f}",
                "",
                "=" * 50,
                "  SENTIMENT ANALYSIS  (bert-base-uncased, 3-class)",
                "=" * 50,
                f"  Test Accuracy  :  {sentiment_metrics['accuracy']:.4f}  ({sentiment_metrics['accuracy'] * 100:.2f}%)",
                f"  Test F1 (wtd)  :  {sentiment_metrics['f1']:.4f}",
                "",
                "-" * 50,
                f"  Dashboard PNG  :  {PERFORMANCE_DASHBOARD_PATH}",
                "-" * 50,
            ]
            self.text_area.insert("end", "\n".join(lines))

        self.text_area.config(state="disabled")


def launch_gui():
    root = tk.Tk()
    root.title("BERT NLP Classifier")
    root.geometry("600x700")
    root.resizable(True, True)
    root.configure(bg="#FAFAFA")

    header = tk.Frame(root, bg="#1F3864", pady=10)
    header.pack(fill="x")
    tk.Label(header, text="BERT NLP Classifier", font=("Arial", 16, "bold"), bg="#1F3864", fg="white").pack()
    tk.Label(
        header,
        text="Sarcasm Detection | Sentiment Analysis",
        font=("Arial", 10),
        bg="#1F3864",
        fg="#A8C4E0",
    ).pack()

    style = ttk.Style()
    style.theme_use("clam")
    style.configure("TNotebook", background="#FAFAFA", borderwidth=0)
    style.configure(
        "TNotebook.Tab",
        font=("Arial", 11, "bold"),
        padding=[14, 6],
        background="#E0E0E0",
        foreground="#333333",
    )
    style.map("TNotebook.Tab", background=[("selected", "#2E75B6")], foreground=[("selected", "white")])

    notebook = ttk.Notebook(root)
    notebook.pack(fill="both", expand=True)

    notebook.add(
        ClassifierTab(
            notebook,
            task_name="sarcasm",
            label_names=["Not Sarcastic", "Sarcastic"],
            heading="Sarcasm Detection",
            action_text="Detect",
        ),
        text="Sarcasm",
    )
    notebook.add(
        ClassifierTab(
            notebook,
            task_name="sentiment",
            label_names=["Negative", "Neutral", "Positive"],
            heading="Sentiment Analysis",
            action_text="Analyze",
        ),
        text="Sentiment",
    )
    notebook.add(MetricsTab(notebook), text="Performance")

    tk.Label(
        root,
        text=f"Device: {DEVICE} | Model: {MODEL_NAME} | Python {sys.version.split()[0]}",
        font=("Arial", 8),
        bg="#E0E0E0",
        fg="#555555",
        anchor="w",
        pady=3,
    ).pack(fill="x", side="bottom")

    root.mainloop()


launch_gui()
